In [88]:
import pandas as pd
import numpy as np
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.metrics import mean_squared_error

#### Q1 - Warm-up with IterativeImputer (Numeric Only)

In [61]:
# Create df where features are correlated

np.random.seed(9009)
n_rows = 500

base = np.random.normal(50,10,n_rows)

df = pd.DataFrame({
    'F1': base,
    'F2': base * 0.5 + np.random.normal(0,5,n_rows),
    'F3': base * 2.1 - np.random.normal(0,3,n_rows),
    'F4': np.random.normal(35,5,n_rows),
    'F5': base * 10 - np.random.normal(0, 2 ,n_rows),
})
df.head().round(2)

,F1,F2,F3,F4,F5
0,56.38,25.85,123.59,33.00,565.44
1,55.97,34.09,116.55,30.53,562.97
2,55.18,23.56,118.29,36.12,550.33
3,61.96,28.58,131.28,36.52,621.52
4,44.07,22.09,92.48,34.11,440.38


In [62]:
# Introduce 15% missingness

col_miss = ['F1', 'F2', 'F3']
miss_rate = 0.15

for col in col_miss:
    mask = np.random.choice([True, False], size=n_rows, p=[miss_rate, 1-miss_rate])
    df.loc[mask, col] = np.nan

df.head().round(2)

,F1,F2,F3,F4,F5
0,NaN,25.85,NaN,33.00,565.44
1,55.97,NaN,116.55,30.53,562.97
2,NaN,23.56,118.29,36.12,550.33
3,NaN,28.58,131.28,36.52,621.52
4,44.07,22.09,92.48,34.11,440.38


In [63]:
# Impute

imputer = IterativeImputer(max_iter=100,random_state=9009)
imputed_df = pd.DataFrame(imputer.fit_transform(df), columns=df.columns)
imputed_df.head().round(2)

,F1,F2,F3,F4,F5
0,56.53,25.85,118.70,33.00,565.44
1,55.97,27.95,116.55,30.53,562.97
2,55.06,23.56,118.29,36.12,550.33
3,62.15,28.58,131.28,36.52,621.52
4,44.07,22.09,92.48,34.11,440.38


In [66]:
# Summary

print("Summary Stats: Before Imputation")
print(df.describe().loc[['mean', 'std']].round(2))

print("\nSummary Stats: After MICE Imputation")
print(imputed_df.describe().loc[['mean', 'std']].round(2))

Summary Stats: Before Imputation
         F1     F2      F3     F4      F5
mean  50.06  25.14  106.10  35.15  502.30
std   10.22   7.14   22.17   4.71  103.38

Summary Stats: After MICE Imputation
         F1     F2      F3     F4      F5
mean  50.22  25.19  105.59  35.15  502.30
std   10.34   6.83   21.98   4.71  103.38


#### Q2 - MAR Pattern + Holdout RMSE

In [103]:
# Created df where X4 depends on X1 & X2

x1 = np.random.normal(50,10,n_rows)
x2 = np.random.normal(70,5,n_rows)
x3 = np.random.normal(100,23,n_rows)
x4 = (0.7 * x1) + (0.3 * x2) + np.random.normal(0,5,n_rows)

df = pd.DataFrame({
    'X1': x1,
    'X2': x2,
    'X3': x3,
    'X4': x4,
})
df.head().round(2)

,X1,X2,X3,X4
0,43.71,70.25,146.25,52.59
1,53.32,73.29,82.26,63.78
2,49.81,63.28,80.91,60.04
3,50.85,71.19,78.31,63.96
4,56.43,75.14,99.29,57.04


In [104]:
# Create MAR

X4_true = df['X4'].copy()

median_x1 = df['X1'].median()
mask = df['X4'] < median_x1
df.loc[mask, 'X4'] = np.nan

print(f'Total Missing Values: {df['X4'].isnull().sum()}')



Total Missing Values: 123


In [105]:
# MICE

imputer = IterativeImputer(max_iter=100,random_state=9009)
imputed_df = pd.DataFrame(imputer.fit_transform(df), columns=df.columns)
imputed_df.head().round(2)

,X1,X2,X3,X4
0,43.71,70.25,146.25,52.59
1,53.32,73.29,82.26,63.78
2,49.81,63.28,80.91,60.04
3,50.85,71.19,78.31,63.96
4,56.43,75.14,99.29,57.04


In [113]:
# RMSE

X4_imputed = imputed_df.loc[mask, 'X4']
X4_true = X4_true[mask]

rmse = np.sqrt(mean_squared_error(X4_true, X4_imputed))
print(f' RMSE: {rmse:.2f}')

 RMSE: 8.43
